<a href="https://colab.research.google.com/github/Udari2002/Image-Processing-Assessment-01/blob/main/practical_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import cv2
import numpy as np
import glob
import os

# 1. Quantitative Metrics Functions [cite: 119-128]
def calculate_dice(segmented, gt):
    intersection = np.logical_and(segmented, gt).sum()
    total = segmented.sum() + gt.sum()
    return (2. * intersection) / total if total > 0 else 1.0

def calculate_jaccard(segmented, gt):
    intersection = np.logical_and(segmented, gt).sum()
    union = np.logical_or(segmented, gt).sum()
    return intersection / union if union > 0 else 1.0

# 2. Student-Defined Vessel Segmentation Pipeline (AI-Free) [cite: 97-102]
def segment_blood_vessels(img):
    # Step 1: Green Channel extraction (Maximizes vessel contrast)
    _, green_ch, _ = cv2.split(img)

    # Step 2: CLAHE for local contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(green_ch)

    # Step 3: Morphological Top-hat to extract small bright structures (vessels)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    tophat = cv2.morphologyEx(enhanced, cv2.MORPH_TOPHAT, kernel)

    # Step 4: Gaussian Smoothing
    blurred = cv2.GaussianBlur(tophat, (3, 3), 0)

    # Step 5: Automated Otsu Thresholding
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Step 6: Median Filtering for noise reduction
    return cv2.medianBlur(thresh, 3)

# 3. Main Processing Loop
base_path = '/content/drive/MyDrive/Computer_vision_Assessment_01/Validation'
image_files = sorted(glob.glob(f"{base_path}/Foundus Images For Validation/*.png"))[:50]
mask_files = sorted(glob.glob(f"{base_path}/Ground Truth For Validation/*.png"))[:50]

if not os.path.exists('Report_Images'): os.makedirs('Report_Images')

all_dice, all_jaccard = [], []
print(f"{'Image Name':<25} | {'Dice Score':<10} | {'Jaccard Index':<10}")
print("-" * 55)

for i in range(len(image_files)):
    img = cv2.imread(image_files[i])
    gt = cv2.imread(mask_files[i], 0)

    # Run the automated pipeline
    result = segment_blood_vessels(img)

    # Binarize and calculate metrics [cite: 115]
    res_bin = (result > 0).astype(np.uint8)
    gt_bin = (gt > 0).astype(np.uint8)

    d_score = calculate_dice(res_bin, gt_bin)
    j_score = calculate_jaccard(res_bin, gt_bin)
    all_dice.append(d_score); all_jaccard.append(j_score)

    print(f"{os.path.basename(image_files[i]):<25} | {d_score:.4f}     | {j_score:.4f}")

    # Save first 10 for the LaTeX report grid [cite: 23]
    if i < 10:
        cv2.imwrite(f'Report_Images/img{i+1}_original.png', img)
        cv2.imwrite(f'Report_Images/img{i+1}_result.png', result)
        cv2.imwrite(f'Report_Images/img{i+1}_gt.png', gt)

print("-" * 55)
print(f"AVERAGE (50 Images)       | {np.mean(all_dice):.4f}     | {np.mean(all_jaccard):.4f}")

Image Name                | Dice Score | Jaccard Index
-------------------------------------------------------
100_A.png                 | 0.0000     | 0.0000
10_A.png                  | 0.0539     | 0.0277
11_A.png                  | 0.0707     | 0.0366
12_A.png                  | 0.0396     | 0.0202
13_A.png                  | 0.0452     | 0.0231
14_A.png                  | 0.0000     | 0.0000
15_A.png                  | 0.0000     | 0.0000
16_A.png                  | 0.0000     | 0.0000
17_A.png                  | 0.0000     | 0.0000
18_A.png                  | 0.0000     | 0.0000
19_A.png                  | 0.0000     | 0.0000
1_A.png                   | 0.0682     | 0.0353
20_A.png                  | 0.0691     | 0.0358
21_A.png                  | 0.0000     | 0.0000
22_A.png                  | 0.0917     | 0.0480
23_A.png                  | 0.0457     | 0.0234
24_A.png                  | 0.0927     | 0.0486
25_A.png                  | 0.0679     | 0.0352
26_A.png                 